# AI Report Generator — Prompt Engineering + RAG

A Google Colab prototype for generating a personalized financial report from transaction, budget, goal, and forecast data.

**Architecture:** financial data → embeddings → ChromaDB retrieval → structured prompt → LLM → report.

The notebook runs locally with **Ollama** when available and falls back to a deterministic template generator so the RAG pipeline can still be tested in Colab.

In [1]:
!pip -q install chromadb sentence-transformers pandas numpy requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

## 1. Imports and sample WealthWise data

In [2]:
import json
import re
import requests
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

# Replace these examples with data returned by the WealthWise Django API.
transactions = pd.DataFrame([
    ['2026-08-01', 'Amazon', 'Shopping', 2499],
    ['2026-08-03', 'Uber', 'Transport', 420],
    ['2026-08-05', 'Swiggy', 'Food', 780],
    ['2026-08-07', 'Netflix', 'Entertainment', 649],
    ['2026-08-09', 'Amazon', 'Shopping', 1299],
    ['2026-08-10', 'Zomato', 'Food', 620],
    ['2026-08-12', 'Uber', 'Transport', 380],
    ['2026-08-14', 'BigBasket', 'Groceries', 1850],
], columns=['date','merchant','category','amount'])

budgets = pd.DataFrame([
    ['Food', 5000, 4200],
    ['Shopping', 4000, 3798],
    ['Transport', 3000, 800],
    ['Entertainment', 2000, 649],
    ['Groceries', 5000, 1850],
], columns=['category','budget','spent'])

goals = pd.DataFrame([
    ['Emergency Fund', 100000, 35000],
    ['Laptop', 80000, 52000],
], columns=['goal','target','saved'])

forecast = pd.DataFrame([
    ['2026-09', 18500],
], columns=['period','predicted_spending'])

display(transactions)
display(budgets)
display(goals)
display(forecast)


,date,merchant,category,amount
0,2026-08-01,Amazon,Shopping,2499
1,2026-08-03,Uber,Transport,420
2,2026-08-05,Swiggy,Food,780
3,2026-08-07,Netflix,Entertainment,649
4,2026-08-09,Amazon,Shopping,1299
5,2026-08-10,Zomato,Food,620
6,2026-08-12,Uber,Transport,380
7,2026-08-14,BigBasket,Groceries,1850


,category,budget,spent
0,Food,5000,4200
1,Shopping,4000,3798
2,Transport,3000,800
3,Entertainment,2000,649
4,Groceries,5000,1850


,goal,target,saved
0,Emergency Fund,100000,35000
1,Laptop,80000,52000


,period,predicted_spending
0,2026-09,18500


## 2. Build RAG documents

In [3]:
documents = []

for _, r in transactions.iterrows():
    documents.append(
        f"Transaction on {r.date}: merchant={r.merchant}, category={r.category}, amount=INR {r.amount:.2f}."
    )

for _, r in budgets.iterrows():
    documents.append(
        f"Budget category={r.category}: budget=INR {r.budget:.2f}, spent=INR {r.spent:.2f}, remaining=INR {r.budget-r.spent:.2f}."
    )

for _, r in goals.iterrows():
    documents.append(
        f"Financial goal={r.goal}: target=INR {r.target:.2f}, saved=INR {r.saved:.2f}, remaining=INR {r.target-r.saved:.2f}."
    )

for _, r in forecast.iterrows():
    documents.append(
        f"Spending forecast for {r.period}: predicted spending=INR {r.predicted_spending:.2f}."
    )

print('RAG documents:', len(documents))
print('\n'.join(documents[:5]))


RAG documents: 16
Transaction on 2026-08-01: merchant=Amazon, category=Shopping, amount=INR 2499.00.
Transaction on 2026-08-03: merchant=Uber, category=Transport, amount=INR 420.00.
Transaction on 2026-08-05: merchant=Swiggy, category=Food, amount=INR 780.00.
Transaction on 2026-08-07: merchant=Netflix, category=Entertainment, amount=INR 649.00.
Transaction on 2026-08-09: merchant=Amazon, category=Shopping, amount=INR 1299.00.


## 3. Create embeddings and ChromaDB vector store

In [4]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, normalize_embeddings=True).tolist()

client = chromadb.Client()
collection = client.get_or_create_collection('wealthwise_financial_context')

collection.upsert(
    ids=[f'doc_{i}' for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings
)

print('Stored documents:', collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored documents: 16


## 4. Retrieve relevant financial context

In [5]:
def retrieve_context(query, top_k=8):
    q_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=q_embedding, n_results=top_k)
    return result['documents'][0]

query = 'Analyze my recent spending, budget performance, financial goals, and expected spending next month.'
context = retrieve_context(query)

print('Retrieved context:\n')
for item in context:
    print('-', item)


Retrieved context:

- Spending forecast for 2026-09: predicted spending=INR 18500.00.
- Budget category=Transport: budget=INR 3000.00, spent=INR 800.00, remaining=INR 2200.00.
- Budget category=Shopping: budget=INR 4000.00, spent=INR 3798.00, remaining=INR 202.00.
- Budget category=Food: budget=INR 5000.00, spent=INR 4200.00, remaining=INR 800.00.
- Budget category=Groceries: budget=INR 5000.00, spent=INR 1850.00, remaining=INR 3150.00.
- Budget category=Entertainment: budget=INR 2000.00, spent=INR 649.00, remaining=INR 1351.00.
- Financial goal=Laptop: target=INR 80000.00, saved=INR 52000.00, remaining=INR 28000.00.
- Financial goal=Emergency Fund: target=INR 100000.00, saved=INR 35000.00, remaining=INR 65000.00.


## 5. Structured report prompt

In [6]:
def build_prompt(context):
    context_text = '\n'.join(f'- {x}' for x in context)
    return f'''You are WealthWise, a personal finance reporting assistant.

Generate a concise financial report using ONLY the supplied financial context.
Do not invent transactions, amounts, trends, or financial facts.
If information is insufficient, explicitly say so.

Return these sections:
1. Executive Summary
2. Spending Analysis
3. Budget Performance
4. Financial Goals
5. Forecast / Outlook
6. Key Risks
7. Actionable Recommendations

Use INR for monetary values. Keep recommendations practical and based on the supplied data.

FINANCIAL CONTEXT:
{context_text}
'''

prompt = build_prompt(context)
print(prompt)


You are WealthWise, a personal finance reporting assistant.

Generate a concise financial report using ONLY the supplied financial context.
Do not invent transactions, amounts, trends, or financial facts.
If information is insufficient, explicitly say so.

Return these sections:
1. Executive Summary
2. Spending Analysis
3. Budget Performance
4. Financial Goals
5. Forecast / Outlook
6. Key Risks
7. Actionable Recommendations

Use INR for monetary values. Keep recommendations practical and based on the supplied data.

FINANCIAL CONTEXT:
- Spending forecast for 2026-09: predicted spending=INR 18500.00.
- Budget category=Transport: budget=INR 3000.00, spent=INR 800.00, remaining=INR 2200.00.
- Budget category=Shopping: budget=INR 4000.00, spent=INR 3798.00, remaining=INR 202.00.
- Budget category=Food: budget=INR 5000.00, spent=INR 4200.00, remaining=INR 800.00.
- Budget category=Groceries: budget=INR 5000.00, spent=INR 1850.00, remaining=INR 3150.00.
- Budget category=Entertainment: budge

## 6. Generate report with Ollama

In [10]:
OLLAMA_URL = 'https://bikini-chosen-bye-collective.trycloudflare.com/api/generate'
OLLAMA_MODEL = 'llama3.2'

def generate_with_ollama(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False},
        timeout=120
    )
    response.raise_for_status()
    return response.json()['response']

try:
    report = generate_with_ollama(prompt)
    print(report)
except requests.exceptions.RequestException as e:
    print(f'Ollama is not reachable from this Colab runtime. Error: {e}')
    print('Use the fallback generator below or connect Colab to a reachable Ollama endpoint.')

**Executive Summary**

This financial report provides an analysis of the current budget categories and financial goals for the period ending 2026-09. The spending forecast for 2026-09 is predicted to be INR 18500.00.

**Spending Analysis**

- Transport: Budget (INR 3000.00), Spent (INR 800.00), Remaining (INR 2200.00)
- Shopping: Budget (INR 4000.00), Spent (INR 3798.00), Remaining (INR 202.00)
- Food: Budget (INR 5000.00), Spent (INR 4200.00), Remaining (INR 800.00)
- Groceries: Budget (INR 5000.00), Spent (INR 1850.00), Remaining (INR 3150.00)
- Entertainment: Budget (INR 2000.00), Spent (INR 649.00), Remaining (INR 1351.00)

**Budget Performance**

The budget performance is as follows:

- Transport: 80% of budget spent
- Shopping: 95.5% of budget spent
- Food: 84% of budget spent
- Groceries: 37% of budget spent
- Entertainment: 32.45% of budget spent

**Financial Goals**

- Laptop: Target (INR 80000.00), Saved (INR 52000.00), Remaining (INR 28000.00)
- Emergency Fund: Target (INR 1

## 7. Colab fallback report generator

In [11]:
total_spending = transactions['amount'].sum()
category_spending = transactions.groupby('category')['amount'].sum().sort_values(ascending=False)
budget_utilization = budgets.assign(utilization=lambda x: x.spent / x.budget * 100)

report_fallback = f'''# WealthWise Financial Report

## Executive Summary
Total recorded spending is INR {total_spending:,.2f}. The highest-spending category is {category_spending.index[0]} at INR {category_spending.iloc[0]:,.2f}.

## Spending Analysis
{category_spending.to_string()}

## Budget Performance
{budget_utilization[['category','utilization']].round(1).to_string(index=False)}

## Financial Goals
{goals[['goal','target','saved']].to_string(index=False)}

## Forecast / Outlook
The current forecast indicates approximately INR {forecast['predicted_spending'].iloc[0]:,.2f} in spending for the next forecast period.

## Key Risks
Monitor categories with high budget utilization and recurring discretionary spending.

## Actionable Recommendations
Review the highest-spending category first, keep high-utilization budgets under review, and compare projected spending with available monthly budget before increasing discretionary spending.
'''

print(report_fallback)


# WealthWise Financial Report

## Executive Summary
Total recorded spending is INR 8,497.00. The highest-spending category is Shopping at INR 3,798.00.

## Spending Analysis
category
Shopping         3798
Groceries        1850
Food             1400
Transport         800
Entertainment     649

## Budget Performance
     category  utilization
         Food         84.0
     Shopping         95.0
    Transport         26.7
Entertainment         32.4
    Groceries         37.0

## Financial Goals
          goal  target  saved
Emergency Fund  100000  35000
        Laptop   80000  52000

## Forecast / Outlook
The current forecast indicates approximately INR 18,500.00 in spending for the next forecast period.

## Key Risks
Monitor categories with high budget utilization and recurring discretionary spending.

## Actionable Recommendations
Review the highest-spending category first, keep high-utilization budgets under review, and compare projected spending with available monthly budget before i

## 8. Export report

In [12]:
final_report = report if 'report' in globals() and report else report_fallback

with open('wealthwise_ai_financial_report.md', 'w', encoding='utf-8') as f:
    f.write(final_report)

print('Saved: wealthwise_ai_financial_report.md')


Saved: wealthwise_ai_financial_report.md


## WealthWise integration

Replace the sample DataFrames with Django API responses. In production, create RAG documents from the user's transactions, budgets, goals, recurring transactions, anomaly scores, and forecasts. Retrieve only the context relevant to the requested report, then pass that context to the LLM using the fixed prompt structure above.